# einops-reduce composite — cx30: reduce to per-row max, then repeat-broadcast for softmax-style subtraction

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-reduce`, `einops-repeat-broadcast`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-reduce"
DD_ATOM_IDS = ["einops-reduce", "einops-repeat-broadcast"]
DD_SUBTOPICS = ["Einops: Reduce", "Einops: Repeat-as-broadcast"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Numerically-stable softmax is the canonical reduce-then-repeat-broadcast pattern. To prevent exp overflow you subtract each row's MAX from every element of that row before exping — i.e. you need to broadcast a `(B,)` per-row-max back across the original `(B, D)` matrix.

Two einops atoms compose here:
  1. `reduce(x, 'b d -> b', 'max')` collapses D into a per-row scalar.
  2. `repeat(row_max, 'b -> b d', d=D)` re-inserts the d-axis as a stride-0 view (no copy).

After the repeat, the shapes match `(B, D)` and broadcasting handles the subtraction. This is identical to `x - x.max(dim=1, keepdim=True).values` — but the einops named pattern makes the reduce / re-broadcast split EXPLICIT.

### Composite Exercise — reduce to per-row max, then repeat-broadcast for softmax-style subtraction

**Atoms exercised together**: `einops-reduce`, `einops-repeat-broadcast`

Implement `cx30_stable_softmax(x)` — numerically stable softmax of an `(B, D)` matrix using the reduce-then-repeat-broadcast composition.

1. **Reduce** to a per-row max with `reduce(x, 'b d -> b', 'max')`. The result has shape `(B,)`.
2. **Repeat-broadcast** the per-row max back to `(B, D)` with `repeat(row_max, 'b -> b d', d=D)`. This is a stride-0 view — no copy.
3. Subtract, `.exp()`, then divide by per-row sum (use the same reduce-then-repeat-broadcast pattern for the denominator).

Return shape `(B, D)`. Cross-check against `torch.nn.functional.softmax(x, dim=1)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx30_stable_softmax(x):
    raise NotImplementedError

def _test_cx30():
    import torch.nn.functional as F

    # Case A: cross-check against F.softmax(dim=1).
    x = t.randn(4, 16)
    out = cx30_stable_softmax(x)
    assert tuple(out.shape) == (4, 16), f'shape: {tuple(out.shape)}'
    ref = F.softmax(x, dim=1)
    assert t.allclose(out, ref, atol=1e-6), f'max diff {(out-ref).abs().max()}'

    # Case B: rows sum to 1.
    row_sums = out.sum(dim=1)
    assert t.allclose(row_sums, t.ones(4), atol=1e-5), f'row sums: {row_sums}'

    # Case C: numerical stability — large positive logits.
    big = t.tensor([[1000.0, 1001.0, 1002.0], [-1000.0, 0.0, 1000.0]])
    out_big = cx30_stable_softmax(big)
    assert t.isfinite(out_big).all(), 'softmax must not overflow on large logits'
    assert t.allclose(out_big.sum(dim=1), t.ones(2), atol=1e-5)
    # Cross-check against F.softmax which is also numerically stable.
    assert t.allclose(out_big, F.softmax(big, dim=1), atol=1e-6)

    # Case D: realistic shape (transformer logits).
    logits = t.randn(8, 50257)
    probs = cx30_stable_softmax(logits)
    assert tuple(probs.shape) == (8, 50257)
    assert t.allclose(probs.sum(dim=1), t.ones(8), atol=1e-4)
    assert t.allclose(probs, F.softmax(logits, dim=1), atol=1e-5)
    # --- atom-coverage: einops.repeat must be used AND produce stride-0 views; plain (B,1)-(B,D) broadcast is not enough ---
    import inspect as _inspect
    _src = _inspect.getsource(cx30_stable_softmax)
    assert 'repeat(' in _src, 'must use einops.repeat to expand reduced rows back to (B, D)'
    assert '.expand(' not in _src and 'expand_as' not in _src, 'must use einops.repeat, not torch.expand'
    assert 'broadcast_to' not in _src, 'must use einops.repeat, not broadcast_to'
    _g = cx30_stable_softmax.__globals__
    _orig_repeat = _g.get('repeat')
    _calls = []
    def _spy_repeat(*a, **kw):
        r = _orig_repeat(*a, **kw)
        _calls.append(r)
        return r
    _g['repeat'] = _spy_repeat
    try:
        cx30_stable_softmax(t.randn(4, 9))
    finally:
        _g['repeat'] = _orig_repeat
    assert len(_calls) >= 1, 'cx30_stable_softmax must call einops.repeat'
    # The d-axis inserted by repeat must be a stride-0 broadcast view (no copy).
    assert any(0 in r.stride() for r in _calls), (
        'einops.repeat output must be a stride-0 broadcast view along the inserted d-axis'
    )

    _dd_passed.add('cx30')

_test_cx30()

<details><summary>Show solution — cx30</summary>

```python
def cx30_stable_softmax(x):
    B, D = x.shape
    # Atom A (einops-reduce): collapse D into per-row max — shape (B,).
    row_max = reduce(x, 'b d -> b', 'max')
    # Atom B (einops-repeat-broadcast): insert d-axis as stride-0 view — shape (B, D).
    row_max_b = repeat(row_max, 'b -> b d', d=D)
    shifted = x - row_max_b
    ex = shifted.exp()
    # Same composition for the denominator: reduce to (B,), then repeat-broadcast to (B, D).
    row_sum = reduce(ex, 'b d -> b', 'sum')
    row_sum_b = repeat(row_sum, 'b -> b d', d=D)
    return ex / row_sum_b
```

The reduce-then-repeat-broadcast pair is the einops-native version of `keepdim=True`. You can fuse the two into a single `reduce(..., 'b d -> b 1', 'max')` call (which is what cx27 does) and let plain broadcasting handle the alignment — but separating them out makes the data flow explicit and keeps the `repeat` step zero-copy (stride-0 view). Either form compiles to the same `expand` under the hood; the choice is about code clarity for the next reader.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx30'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx30',
        'subtopics': ["Einops: Reduce", "Einops: Repeat-as-broadcast"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()